# Preparación, limpieza y estructuración de datos — Pokémon por generaciones

**Autor:** Brandon Maldonado Mesa · Ficha 3169901

Este cuaderno continúa el diagnóstico de calidad (`data_quality.ipynb`) y aplica técnicas de limpieza, filtrado, agrupación y estructuración con Pandas al proyecto individual.

**Pregunta analítica:** ¿cómo han cambiado las estadísticas base promedio de los Pokémon (HP, ataque, defensa, ataque especial, defensa especial y velocidad) entre generaciones, y cuál de ellas ha tenido el mayor cambio?

**Pregunta orientadora:** ¿cómo preparar y estructurar los datos del proyecto individual para obtener información confiable que permita responder la pregunta analítica?

**Contenido**

1. Limpieza y estandarización (Lección 2)
2. Filtrado, ordenamiento y agrupación (Lección 3)
3. Estructuración y validación de los datos (Lección 4)

## Punto de partida: resultados del diagnóstico

El diagnóstico de calidad (`reports/data_quality.md`) concluyó que el dataset está completo y es consistente, y dejó estas acciones de preparación pendientes, que este cuaderno aborda:

1. Mantener los nulos de `tipo_2` y definir una categoría explícita solo en los análisis por tipo.
2. Listar los registros atípicos en `hp`, `defensa` y `defensa_especial` y decidir su tratamiento.
3. Crear una columna auxiliar `especial` (legendario o mítico) para calcular promedios con y sin estos casos.
4. Incluir el número de Pokémon por generación en las tablas de resultados.
5. Documentar las limitaciones: una sola fuente, una sola forma por Pokémon y estadísticas vigentes.

## 0. Preparación del entorno

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

RUTA = Path('../data/raw/pokemon_pokeapi.csv')
SALIDA = Path('../data/processed')
SALIDA.mkdir(parents=True, exist_ok=True)

ESTADISTICAS = ['hp', 'ataque', 'defensa', 'ataque_especial', 'defensa_especial', 'velocidad']
TIPOS_VALIDOS = ['normal', 'fire', 'water', 'electric', 'grass', 'ice', 'fighting', 'poison',
                 'ground', 'flying', 'psychic', 'bug', 'rock', 'ghost', 'dragon', 'dark',
                 'steel', 'fairy']
print(pd.__version__)

3.0.6


# Lección 2: limpieza y estandarización

## 1.1 Carga y estado inicial

In [2]:
pokemon_crudo = pd.read_csv(RUTA)
print(pokemon_crudo.shape)
pokemon_crudo.head(3)

(1025, 14)


,id,nombre,generacion,tipo_1,tipo_2,legendario,mitico,hp,ataque,defensa,ataque_especial,defensa_especial,velocidad,total
0,1,bulbasaur,1,grass,poison,False,False,45,49,49,65,65,45,318
1,2,ivysaur,1,grass,poison,False,False,60,62,63,80,80,60,405
2,3,venusaur,1,grass,poison,False,False,80,82,83,100,100,80,525


## 1.2 Duplicados

In [3]:
print('Filas duplicadas:', pokemon_crudo.duplicated().sum())
print('IDs repetidos:', pokemon_crudo['id'].duplicated().sum())
print('Nombres repetidos:', pokemon_crudo['nombre'].duplicated().sum())

# Duplicados implícitos: mismo nombre después de estandarizar
nombres_normalizados = pokemon_crudo['nombre'].str.strip().str.lower()
print('Nombres repetidos tras normalizar:', nombres_normalizados.duplicated().sum())

Filas duplicadas: 0
IDs repetidos: 0
Nombres repetidos: 0
Nombres repetidos tras normalizar: 0


**Resultado:** no hay filas, IDs ni nombres duplicados, ni siquiera después de normalizar el texto. **Decisión:** el paso de eliminación de duplicados queda en el pipeline como protección, pero no elimina ninguna fila.

## 1.3 Valores faltantes

In [4]:
nulos = pokemon_crudo.isna().sum()
print(nulos[nulos > 0])
print('tipo_1 sin nulos:', pokemon_crudo['tipo_1'].notna().all())

un_solo_tipo = pokemon_crudo['tipo_2'].isna()
print('Pokémon de un solo tipo:', un_solo_tipo.sum(), '| de dos tipos:', (~un_solo_tipo).sum())

tipo_2    499
dtype: int64
tipo_1 sin nulos: True
Pokémon de un solo tipo: 499 | de dos tipos: 526


**Interpretación:** los 499 nulos de `tipo_2` corresponden a Pokémon de un solo tipo (`tipo_1` nunca es nulo y 499 + 526 = 1025). Es una ausencia con significado, no un error de captura.

**Decisión: conservar el nulo.** Rellenarlo con un texto cambiaría el significado de la columna; si se analiza por tipo secundario, se usará una categoría explícita como `'ninguno'` solo en una copia.

## 1.4 Texto y categorías

In [5]:
for columna in ['nombre', 'tipo_1', 'tipo_2']:
    s = pokemon_crudo[columna].dropna()
    print(f'{columna:7s}',
          '| espacios sobrantes:', (s != s.str.strip()).sum(),
          '| espacios dobles:', s.str.contains('  ', regex=False).sum(),
          '| con mayúsculas:', (s != s.str.lower()).sum())

tipos_usados = set(pokemon_crudo['tipo_1'].dropna()) | set(pokemon_crudo['tipo_2'].dropna())
print('Tipos distintos:', len(tipos_usados))
print('Tipos fuera de la lista oficial:', sorted(tipos_usados - set(TIPOS_VALIDOS)))
print('Nombres con guion (formas o nombres compuestos):', pokemon_crudo['nombre'].str.contains('-').sum())

nombre  | espacios sobrantes: 0 | espacios dobles: 0 | con mayúsculas: 0
tipo_1  | espacios sobrantes: 0 | espacios dobles: 0 | con mayúsculas: 0
tipo_2  | espacios sobrantes: 0 | espacios dobles: 0 | con mayúsculas: 0
Tipos distintos: 18
Tipos fuera de la lista oficial: []
Nombres con guion (formas o nombres compuestos): 76


**Interpretación:** las columnas de texto ya están estandarizadas (minúsculas, sin espacios sobrantes) y los 18 tipos coinciden con la lista oficial. De los 76 nombres con guion, algunos son nombres compuestos (`ho-oh`, `mr-mime`) y otros llevan el sufijo de la forma por defecto del Pokémon (`deoxys-normal`, `pyroar-male`). Se conservan tal como vienen de la API, porque el análisis usa `id` como identificador.

**Decisión:** la función `estandarizar_texto` queda en el pipeline como protección para datos nuevos, sin cambiar valores en este dataset.

## 1.5 Tipos de dato

In [6]:
print(pokemon_crudo.dtypes)
print('Memoria:', round(pokemon_crudo.memory_usage(deep=True).sum() / 1e3, 1), 'KB')

id                  int64
nombre                str
generacion          int64
tipo_1                str
tipo_2                str
legendario           bool
mitico               bool
hp                  int64
ataque              int64
defensa             int64
ataque_especial     int64
defensa_especial    int64
velocidad           int64
total               int64
dtype: object
Memoria: 234.7 KB


**Interpretación:** los tipos coinciden con el significado de cada columna, pero son más grandes de lo necesario (`int64` para valores que no pasan de 255). Se reducirán a `int16` e `int8`, y `tipo_1` y `tipo_2` pasarán a `category`.

## 1.6 Funciones de limpieza y pipeline con informe de cambios

Cada paso es una función que recibe un DataFrame y devuelve otro, y se encadenan con `pipe`. La función `anotar` toma una "foto" de la tabla después de cada paso, sin modificarla, y `validar_datos` comprueba con `assert` las reglas que deben cumplir los datos limpios.

In [7]:
registro = []

def anotar(df, paso):
    registro.append({
        'paso': paso,
        'filas': len(df),
        'columnas': df.shape[1],
        'nulos': int(df.isna().sum().sum()),
        'memoria_kb': round(df.memory_usage(deep=True).sum() / 1e3, 1),
    })
    return df

def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def estandarizar_texto(df):
    df = df.copy()
    for columna in ['nombre', 'tipo_1', 'tipo_2']:
        df[columna] = df[columna].str.strip().str.lower()
    return df

def ajustar_tipos(df):
    tipos = {'id': 'int16', 'generacion': 'int8', 'total': 'int16',
             'tipo_1': 'category', 'tipo_2': 'category'}
    tipos.update({estadistica: 'int16' for estadistica in ESTADISTICAS})
    return df.astype(tipos)

def agregar_especial(df):
    df = df.copy()
    df['especial'] = df['legendario'] | df['mitico']
    return df

def validar_datos(df):
    assert len(df) == 1025, 'número de filas inesperado'
    assert df['id'].tolist() == list(range(1, 1026)), 'IDs no consecutivos'
    assert df['nombre'].is_unique, 'nombres repetidos'
    assert df['generacion'].between(1, 9).all(), 'generación fuera de rango'
    assert df[ESTADISTICAS].gt(0).all().all(), 'estadística menor o igual a 0'
    assert df[ESTADISTICAS].le(255).all().all(), 'estadística mayor a 255'
    assert (df[ESTADISTICAS].sum(axis=1) == df['total']).all(), 'total distinto de la suma'
    assert not (df['legendario'] & df['mitico']).any(), 'legendario y mítico a la vez'
    assert df['tipo_1'].notna().all(), 'tipo_1 con nulos'
    assert df['tipo_1'].isin(TIPOS_VALIDOS).all(), 'tipo_1 inválido'
    assert (df['tipo_2'].isna() | df['tipo_2'].isin(TIPOS_VALIDOS)).all(), 'tipo_2 inválido'
    assert (df['tipo_2'].dropna() != df.loc[df['tipo_2'].notna(), 'tipo_1']).all(), 'tipo_2 igual a tipo_1'
    return df

In [8]:
pokemon = (pokemon_crudo
    .pipe(anotar, '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, '1 sin duplicados')
    .pipe(estandarizar_texto)
    .pipe(anotar, '2 texto estandarizado')
    .pipe(ajustar_tipos)
    .pipe(anotar, '3 tipos')
    .pipe(agregar_especial)
    .pipe(anotar, '4 columna especial')
    .pipe(validar_datos))

informe_limpieza = pd.DataFrame(registro)
informe_limpieza

,paso,filas,columnas,nulos,memoria_kb
0,0 crudo,1025,14,499,234.7
1,1 sin duplicados,1025,14,499,234.7
2,2 texto estandarizado,1025,14,499,234.7
3,3 tipos,1025,14,499,82.0
4,4 columna especial,1025,15,499,83.0


**Interpretación:** el pipeline no eliminó filas ni cambió valores. Ajustó los tipos de dato (la memoria baja a poco más de un tercio), agregó la columna `especial` (legendario o mítico) y superó todas las validaciones. Los 499 nulos de `tipo_2` se mantienen a propósito.

## 1.7 Estado inicial y final de los datos

In [9]:
def metricas_calidad(df):
    estadisticas = df[ESTADISTICAS]
    tipos_invalidos = (~df['tipo_1'].isin(TIPOS_VALIDOS)).sum() + \
                      (df['tipo_2'].notna() & ~df['tipo_2'].isin(TIPOS_VALIDOS)).sum()
    return {
        'filas': len(df),
        'columnas': df.shape[1],
        'nulos_totales': int(df.isna().sum().sum()),
        'filas_duplicadas': int(df.duplicated().sum()),
        'ids_repetidos': int(df['id'].duplicated().sum()),
        'total_distinto_de_suma': int((estadisticas.sum(axis=1) != df['total']).sum()),
        'estadisticas_fuera_de_rango': int(((estadisticas <= 0) | (estadisticas > 255)).sum().sum()),
        'generacion_fuera_de_rango': int((~df['generacion'].between(1, 9)).sum()),
        'tipos_invalidos': int(tipos_invalidos),
    }

comparacion_calidad = pd.DataFrame({'antes': metricas_calidad(pokemon_crudo),
                                    'despues': metricas_calidad(pokemon)}).astype(object)
comparacion_calidad.loc['memoria_kb'] = [informe_limpieza['memoria_kb'].iloc[0],
                                         informe_limpieza['memoria_kb'].iloc[-1]]
comparacion_calidad

,antes,despues
filas,1025,1025
columnas,14,15
nulos_totales,499,499
filas_duplicadas,0,0
ids_repetidos,0,0
total_distinto_de_suma,0,0
estadisticas_fuera_de_rango,0,0
generacion_fuera_de_rango,0,0
tipos_invalidos,0,0
memoria_kb,234.7,83.0


**Comparación antes y después:** las verificaciones de calidad dan cero casos antes y después (el dataset de origen ya era consistente, como concluyó el diagnóstico). Los cambios reales son la reducción de memoria (a poco más de un tercio), los tipos de dato ajustados y la columna nueva `especial`.

In [10]:
pd.DataFrame({'antes': pokemon_crudo.dtypes.astype(str),
              'despues': pokemon.dtypes.astype(str)}).fillna('(columna nueva)')

,antes,despues
ataque,int64,int16
ataque_especial,int64,int16
defensa,int64,int16
defensa_especial,int64,int16
especial,(columna nueva),bool
generacion,int64,int8
hp,int64,int16
id,int64,int16
legendario,bool,bool
mitico,bool,bool


# Lección 3: filtrado, ordenamiento y agrupación

Se aplican técnicas de análisis sobre el DataFrame limpio (`pokemon`) para responder la pregunta analítica: ¿cómo han cambiado las estadísticas base promedio entre generaciones y cuál ha tenido el mayor cambio?

## 2.1 Filtros

In [11]:
# Condición booleana: Pokémon que no son legendarios ni míticos
base = pokemon[~pokemon['especial']]
print('Pokémon base (sin legendarios ni míticos):', len(base))

# query(): primera generación con total alto
print('Generación 1 con total >= 500:', len(pokemon.query('generacion == 1 and total >= 500')))

# isin(): tipos principales de los iniciales clásicos
print('tipo_1 fuego, agua o planta:', pokemon['tipo_1'].isin(['fire', 'water', 'grass']).sum())

# between(): total entre 600 y 720
muy_fuertes = pokemon[pokemon['total'].between(600, 720)]
print('Pokémon con total entre 600 y 720:', len(muy_fuertes))
print('De ellos, legendarios o míticos:', int(muy_fuertes['especial'].sum()))

Pokémon base (sin legendarios ni míticos): 931
Generación 1 con total >= 500: 33
tipo_1 fuego, agua o planta: 303
Pokémon con total entre 600 y 720: 60
De ellos, legendarios o míticos: 48


**Interpretación:** hay 931 Pokémon que no son legendarios ni míticos. De los 60 con total entre 600 y 720, 48 son legendarios o míticos y 12 no lo son. Estos filtros muestran que la columna `especial` separa bien los casos extremos que podrían sesgar los promedios.

## 2.2 Ordenamiento y selección de registros relevantes

In [12]:
columnas = ['id', 'nombre', 'generacion', 'total', 'especial']
print('Mayor total entre los Pokémon base')
print(base.nlargest(5, 'total')[columnas].to_string(index=False))
print()
print('Mayor HP entre todos los Pokémon')
print(pokemon.sort_values('hp', ascending=False)[['nombre', 'generacion', 'hp']].head(5).to_string(index=False))

Mayor total entre los Pokémon base
 id    nombre  generacion  total  especial
289   slaking           3    670     False
149 dragonite           1    600     False
248 tyranitar           2    600     False
373 salamence           3    600     False
376 metagross           3    600     False

Mayor HP entre todos los Pokémon
   nombre  generacion  hp
  blissey           2 255
  chansey           1 250
 guzzlord           7 223
regidrago           8 200
wobbuffet           2 190


**Interpretación:** entre los Pokémon base, Slaking (generación 3) tiene el mayor total (670) y le siguen varios con 600 (Dragonite, Tyranitar, Salamence, Metagross). El HP más alto es el de Blissey (255), seguido de Chansey (250) y Guzzlord (223): son los mismos extremos que detectó el diagnóstico.

## 2.3 Resumen por generación con `groupby` y `agg`

In [13]:
resumen = (pokemon
    .groupby('generacion')
    .agg(pokemon=('id', 'size'),
         especiales=('especial', 'sum'),
         total_promedio=('total', 'mean'),
         hp=('hp', 'mean'),
         ataque=('ataque', 'mean'),
         defensa=('defensa', 'mean'),
         ataque_especial=('ataque_especial', 'mean'),
         defensa_especial=('defensa_especial', 'mean'),
         velocidad=('velocidad', 'mean')))

resumen_base = (base
    .groupby('generacion')
    .agg(pokemon=('id', 'size'),
         total_promedio=('total', 'mean'),
         hp=('hp', 'mean'),
         ataque=('ataque', 'mean'),
         defensa=('defensa', 'mean'),
         ataque_especial=('ataque_especial', 'mean'),
         defensa_especial=('defensa_especial', 'mean'),
         velocidad=('velocidad', 'mean')))

print('Todos los Pokémon')
print(resumen.round(1).to_string())
print()
print('Sin legendarios ni míticos')
print(resumen_base.round(1).to_string())

Todos los Pokémon
            pokemon  especiales  total_promedio    hp  ataque  defensa  ataque_especial  defensa_especial  velocidad
generacion                                                                                                          
1               151           5           407.6  64.2    72.9     68.2             67.1              66.1       69.1
2               100           6           407.2  71.0    68.3     69.7             64.5              72.3       61.4
3               135          10           403.7  65.7    73.1     69.0             67.9              66.5       61.6
4               107          14           445.6  73.1    80.2     75.1             73.3              74.4       69.5
5               156          13           425.8  70.3    81.0     71.2             69.2              67.3       66.6
6                72           6           429.3  68.9    72.5     75.1             72.5              74.6       65.7
7                88          16           449.

**Interpretación:** el total promedio sube de 407,6 (generación 1) a 457,4 (generación 9) con todos los Pokémon, y de 400,9 a 444,5 sin legendarios ni míticos. Las generaciones tienen tamaños distintos (de 72 a 156 Pokémon), por eso cada promedio va junto a su número de Pokémon.

## 2.4 Indicadores

Se calculan cuatro indicadores relacionados con la pregunta analítica:

1. **Cambio entre la generación 1 y la 9** de cada estadística (diferencia y porcentaje).
2. **Tendencia por generación:** pendiente de una recta ajustada a los promedios de las nueve generaciones.
3. **Proporción de legendarios y míticos** en cada generación.
4. **Contribución de los legendarios y míticos** al cambio del total promedio.

In [14]:
def cambio_por_estadistica(tabla):
    primera = tabla.loc[1, ESTADISTICAS]
    ultima = tabla.loc[9, ESTADISTICAS]
    generaciones = tabla.index.to_numpy(dtype=float)
    pendiente = pd.Series({e: np.polyfit(generaciones, tabla[e], 1)[0] for e in ESTADISTICAS})
    indicador = pd.DataFrame({
        'gen_1': primera,
        'gen_9': ultima,
        'cambio': ultima - primera,
        'cambio_pct': (ultima / primera - 1) * 100,
        'pendiente_por_gen': pendiente,
    })
    return indicador.sort_values('cambio', ascending=False).round(2)

indicador_1_2_todos = cambio_por_estadistica(resumen)
indicador_1_2_base = cambio_por_estadistica(resumen_base)

print('Indicadores 1 y 2 — todos los Pokémon')
print(indicador_1_2_todos.to_string())
print()
print('Indicadores 1 y 2 — sin legendarios ni míticos')
print(indicador_1_2_base.to_string())

Indicadores 1 y 2 — todos los Pokémon
                  gen_1  gen_9  cambio  cambio_pct  pendiente_por_gen
hp                64.21  77.39   13.18       20.53               1.08
ataque            72.91  82.42    9.51       13.04               1.63
defensa           68.23  76.78    8.56       12.54               1.07
defensa_especial  66.09  72.47    6.39        9.67               0.56
velocidad         69.07  75.45    6.38        9.24               0.86
ataque_especial   67.14  72.87    5.73        8.53               0.96

Indicadores 1 y 2 — sin legendarios ni míticos
                  gen_1  gen_9  cambio  cambio_pct  pendiente_por_gen
hp                63.15  75.83   12.68       20.08               0.90
ataque            72.09  80.64    8.55       11.86               1.49
defensa           67.38  74.28    6.90       10.24               0.98
ataque_especial   65.34  71.09    5.76        8.81               0.92
velocidad         67.97  73.06    5.09        7.49               0.63
defe

**Interpretación (indicadores 1 y 2):**

- **HP es la estadística con mayor cambio entre la generación 1 y la 9:** de 64,2 a 77,4 (+13,2 puntos, +20,5 %). Sigue primero al excluir legendarios y míticos (+12,7). Su promedio en la generación 9 es el más alto de las nueve generaciones.
- **Ataque (+9,5) y defensa (+8,6)** le siguen, y mantienen el segundo y el tercer puesto sin legendarios ni míticos.
- **Ataque es la estadística con mayor tendencia:** sube 1,63 puntos por generación frente a 1,08 de HP.
- La respuesta depende de la definición de cambio: por diferencia entre la primera y la última generación gana HP; por tendencia a lo largo de las nueve generaciones gana ataque.

In [15]:
indicador_3 = (resumen[['pokemon', 'especiales']]
               .assign(especiales_pct=lambda t: (t['especiales'] / t['pokemon'] * 100).round(1)))
print('Indicador 3 — proporción de legendarios y míticos')
print(indicador_3.to_string())

cambio_total_todos = resumen.loc[9, 'total_promedio'] - resumen.loc[1, 'total_promedio']
cambio_total_base = resumen_base.loc[9, 'total_promedio'] - resumen_base.loc[1, 'total_promedio']
print()
print('Indicador 4 — cambio del total promedio (gen 9 menos gen 1)')
print('Todos los Pokémon:           ', round(cambio_total_todos, 1))
print('Sin legendarios ni míticos:  ', round(cambio_total_base, 1))
print('Aporte de legendarios y míticos:', round(cambio_total_todos - cambio_total_base, 1))

Indicador 3 — proporción de legendarios y míticos
            pokemon  especiales  especiales_pct
generacion                                     
1               151           5             3.3
2               100           6             6.0
3               135          10             7.4
4               107          14            13.1
5               156          13             8.3
6                72           6             8.3
7                88          16            18.2
8                96          12            12.5
9               120          12            10.0

Indicador 4 — cambio del total promedio (gen 9 menos gen 1)
Todos los Pokémon:            49.7
Sin legendarios ni míticos:   43.5
Aporte de legendarios y míticos: 6.2


**Interpretación (indicadores 3 y 4):** la proporción de legendarios y míticos sube de 3,3 % en la generación 1 a 18,2 % en la generación 7 (16 de 88) y queda en 10,0 % en la 9. Aun así, del aumento de 49,7 puntos del total promedio, solo 6,2 se deben a ellos: unos 43,5 puntos (alrededor del 87 %) se mantienen al excluirlos.

**Sensibilidad a valores extremos.** Se repite el cambio entre la generación 9 y la 1 usando la mediana de los Pokémon base.

In [16]:
mediana_base = base.groupby('generacion')[ESTADISTICAS].median()
(mediana_base.loc[9] - mediana_base.loc[1]).sort_values(ascending=False)

hp                  13.0
velocidad            9.0
defensa              5.5
ataque               5.0
ataque_especial      2.5
defensa_especial     2.5
dtype: float64

**Interpretación:** con la mediana, HP sigue primero (+13), pero velocidad pasa al segundo puesto (+9) y ataque baja a +5. El primer lugar de HP es estable; el orden de las demás estadísticas cambia según la medida, así que sus diferencias son pequeñas.

## 2.5 Tabla dinámica

In [17]:
dinamica_total = pokemon.pivot_table(index='generacion', columns='especial',
                                     values='total', aggfunc=['mean', 'count']).round(1)
dinamica_total.columns = ['_'.join(map(str, columna)) for columna in dinamica_total.columns]
dinamica_total

,mean_False,mean_True,count_False,count_True
generacion,,,,
1,400.9,604.0,146,5
2,393.8,616.7,94,6
3,386.7,616.0,125,10
4,419.8,616.4,93,14
5,409.2,607.7,143,13
6,411.4,626.7,66,6
7,429.2,540.2,72,16
8,419.3,578.8,84,12
9,444.5,573.8,108,12


**Interpretación:** el total promedio de los legendarios y míticos va de 604 a 627 en las generaciones 1 a 6 y baja a 540 (generación 7), 579 (generación 8) y 574 (generación 9). Es decir, el aumento del total no viene de que estos casos sean más fuertes, sino de los Pokémon base: de 400,9 en la generación 1 a 444,5 en la 9.

## 2.6 Categorización del poder (`total`)

In [18]:
limites = [0, 300, 400, 500, 600, 721]
etiquetas = ['bajo', 'medio', 'alto', 'muy_alto', 'elite']

categorias = pd.cut(pokemon['total'], bins=limites, labels=etiquetas, right=False)
print(categorias.value_counts().reindex(etiquetas))

distribucion = (pd.crosstab(pokemon['generacion'], categorias, normalize='index') * 100).round(1)
distribucion

total
bajo        146
medio       240
alto        329
muy_alto    250
elite        60
Name: count, dtype: int64

total,bajo,medio,alto,muy_alto,elite
generacion,,,,,
1,12.6,33.1,32.5,19.9,2.0
2,18.0,21.0,36.0,21.0,4.0
3,21.5,21.5,40.0,9.6,7.4
4,11.2,22.4,28.0,29.0,9.3
5,10.9,30.8,34.6,18.6,5.1
6,9.7,30.6,27.8,22.2,9.7
7,12.5,15.9,34.1,28.4,9.1
8,17.7,13.5,28.1,35.4,5.2
9,13.3,15.8,24.2,42.5,4.2


**Interpretación:** la distribución del poder se desplaza hacia arriba. La categoría `muy_alto` (total de 500 a 599) pasa de 19,9 % de los Pokémon en la generación 1 a 42,5 % en la 9, mientras que `medio` (300 a 399) baja de 33,1 % a 15,8 %. La categoría `elite` (600 o más) varía entre 2,0 % y 9,7 % sin una tendencia clara.

## Conclusión de la Lección 3

- Las seis estadísticas base promedio subieron entre la generación 1 y la 9. **HP es la que más cambió** (+13,2 puntos, +20,5 %), y **ataque es la que más sube por generación** (+1,63 por generación).
- El aumento no se debe principalmente a los legendarios y míticos: sin ellos, el total promedio sube 43,5 puntos de los 49,7 que se ven con todos.
- El poder de los Pokémon se concentra cada vez más en la franja de 500 a 599 de total.
- Estos resultados son descriptivos: las generaciones tienen tamaños distintos y no se aplicó ninguna prueba estadística.

# Lección 4: estructuración y validación de los datos

## 3.1 Relaciones entre datasets

In [19]:
archivos_originales = sorted(archivo.name for archivo in Path('../data/raw').glob('*.csv'))
print('Datasets originales:', archivos_originales)
print('Clave `id` única:', pokemon['id'].is_unique)
print('Filas:', len(pokemon), '| una fila por Pokémon:', pokemon['id'].nunique() == len(pokemon))

Datasets originales: ['pokemon_pokeapi.csv']
Clave `id` única: True
Filas: 1025 | una fila por Pokémon: True


**Resultado:** hay un único dataset original, con la clave `id` única y una fila por Pokémon.

**Justificación: no se combinan tablas.** El proyecto usa un solo dataset (`pokemon_pokeapi.csv`), con una fila por Pokémon y todas las variables de la pregunta analítica en la misma tabla. No existe una segunda tabla con una relación real que unir mediante una clave, y crear una tabla de apoyo solo para hacer un `merge` sería una combinación artificial. La segunda fuente prevista en el diagnóstico (Kaggle o Bulbapedia) serviría para contrastar valores, no para ampliar columnas.

En lugar de combinar, se aplican técnicas de **estructuración** sobre el dataset único: columnas derivadas (`doble_tipo`, `categoria_poder`), un formato largo con `melt` para las seis estadísticas y tablas de resultados listas para las siguientes etapas.

## 3.2 Estructuración

**Dataset consolidado.** Se agregan dos columnas derivadas al DataFrame limpio, con funciones encadenadas con `pipe`.

In [20]:
def agregar_doble_tipo(df):
    df = df.copy()
    df['doble_tipo'] = df['tipo_2'].notna()
    return df

def agregar_categoria_poder(df):
    df = df.copy()
    df['categoria_poder'] = pd.cut(df['total'], bins=limites, labels=etiquetas, right=False)
    return df

pokemon_final = (pokemon
    .pipe(agregar_doble_tipo)
    .pipe(agregar_categoria_poder))

print(pokemon_final.shape)
print(pokemon_final.dtypes)
pokemon_final.head(3)

(1025, 17)
id                     int16
nombre                   str
generacion              int8
tipo_1              category
tipo_2              category
legendario              bool
mitico                  bool
hp                     int16
ataque                 int16
defensa                int16
ataque_especial        int16
defensa_especial       int16
velocidad              int16
total                  int16
especial                bool
doble_tipo              bool
categoria_poder     category
dtype: object


,id,nombre,generacion,tipo_1,tipo_2,legendario,mitico,hp,ataque,defensa,ataque_especial,defensa_especial,velocidad,total,especial,doble_tipo,categoria_poder
0,1,bulbasaur,1,grass,poison,False,False,45,49,49,65,65,45,318,False,True,medio
1,2,ivysaur,1,grass,poison,False,False,60,62,63,80,80,60,405,False,True,alto
2,3,venusaur,1,grass,poison,False,False,80,82,83,100,100,80,525,False,True,muy_alto


**Interpretación:** el dataset consolidado tiene 1025 filas y 17 columnas (las 14 originales más `especial`, `doble_tipo` y `categoria_poder`). Los nulos de `tipo_2` se conservan, y `doble_tipo` los expresa como indicador sin modificarlos.

**Formato largo.** Las seis estadísticas se pasan de columnas a filas con `melt`, el formato que prefieren `groupby` y las librerías de gráficos.

In [21]:
estadisticas_largo = pokemon_final.melt(
    id_vars=['id', 'nombre', 'generacion', 'especial'],
    value_vars=ESTADISTICAS,
    var_name='estadistica', value_name='valor')
estadisticas_largo['estadistica'] = pd.Categorical(estadisticas_largo['estadistica'],
                                                   categories=ESTADISTICAS, ordered=True)
estadisticas_largo['valor'] = estadisticas_largo['valor'].astype('int16')

print(estadisticas_largo.shape)
estadisticas_largo.head(3)

(6150, 6)


,id,nombre,generacion,especial,estadistica,valor
0,1,bulbasaur,1,False,hp,45
1,2,ivysaur,1,False,hp,60
2,3,venusaur,1,False,hp,80


**Interpretación:** el formato largo tiene 6 150 filas (1 025 Pokémon por 6 estadísticas), una por cada medición.

**Promedios por generación en formato largo**, para todos los Pokémon y sin legendarios ni míticos.

In [22]:
def promedios_largo(df, grupo):
    return (df.groupby(['generacion', 'estadistica'], observed=True)
              .agg(pokemon=('id', 'nunique'), promedio=('valor', 'mean'))
              .round(2).reset_index().assign(grupo=grupo))

promedios_generacion = pd.concat([
    promedios_largo(estadisticas_largo, 'todos'),
    promedios_largo(estadisticas_largo[~estadisticas_largo['especial']], 'sin_especiales'),
], ignore_index=True)

print(promedios_generacion.shape)
promedios_generacion.head(6)

(108, 5)


,generacion,estadistica,pokemon,promedio,grupo
0,1,hp,151,64.21,todos
1,1,ataque,151,72.91,todos
2,1,defensa,151,68.23,todos
3,1,ataque_especial,151,67.14,todos
4,1,defensa_especial,151,66.09,todos
5,1,velocidad,151,69.07,todos


**Interpretación:** la tabla tiene 108 filas (9 generaciones por 6 estadísticas por 2 grupos). Apilar con `concat` los dos grupos no combina datasets distintos: son dos resúmenes del mismo dataset, uno con todos los Pokémon y otro sin legendarios ni míticos.

**Indicadores de cambio** (indicadores 1 y 2 de la Lección 3), en una sola tabla.

In [23]:
indicadores_cambio = (pd.concat([indicador_1_2_todos, indicador_1_2_base],
                                keys=['todos', 'sin_especiales'], names=['grupo', 'estadistica'])
                        .reset_index())
indicadores_cambio

,grupo,estadistica,gen_1,gen_9,cambio,cambio_pct,pendiente_por_gen
0,todos,hp,64.21,77.39,13.18,20.53,1.08
1,todos,ataque,72.91,82.42,9.51,13.04,1.63
2,todos,defensa,68.23,76.78,8.56,12.54,1.07
3,todos,defensa_especial,66.09,72.47,6.39,9.67,0.56
4,todos,velocidad,69.07,75.45,6.38,9.24,0.86
5,todos,ataque_especial,67.14,72.87,5.73,8.53,0.96
6,sin_especiales,hp,63.15,75.83,12.68,20.08,0.90
7,sin_especiales,ataque,72.09,80.64,8.55,11.86,1.49
8,sin_especiales,defensa,67.38,74.28,6.90,10.24,0.98
9,sin_especiales,ataque_especial,65.34,71.09,5.76,8.81,0.92


## 3.3 Validación de integridad y consistencia

In [24]:
REFERENCIA_POR_GENERACION = {1: 151, 2: 100, 3: 135, 4: 107, 5: 156, 6: 72, 7: 88, 8: 96, 9: 120}

# Dataset consolidado
assert len(pokemon_final) == 1025
assert pokemon_final['id'].is_unique
assert pokemon_final['generacion'].value_counts().sort_index().to_dict() == REFERENCIA_POR_GENERACION
assert pokemon_final['categoria_poder'].notna().all()
assert pokemon_final['doble_tipo'].sum() == 526
assert pokemon_final['tipo_2'].isna().sum() == 499          # los nulos de tipo_2 se conservan
assert (pokemon_final['doble_tipo'] == pokemon_final['tipo_2'].notna()).all()

# Formato largo: misma información que el formato ancho
assert len(estadisticas_largo) == 1025 * len(ESTADISTICAS)
assert estadisticas_largo['valor'].notna().all()
suma_por_id = estadisticas_largo.groupby('id')['valor'].sum()
assert (suma_por_id.values == pokemon_final.sort_values('id')['total'].values).all()

# Promedios: el formato largo reproduce las tablas de la Lección 3
for estadistica in ESTADISTICAS:
    en_largo = (promedios_generacion[(promedios_generacion['grupo'] == 'todos')
                                     & (promedios_generacion['estadistica'] == estadistica)]
                .set_index('generacion')['promedio'])
    assert np.allclose(en_largo.values, resumen[estadistica].round(2).values), estadistica
assert promedios_generacion.groupby(['grupo', 'estadistica'], observed=True)['pokemon'].sum().min() == 931
print('Validaciones superadas')

Validaciones superadas


**Interpretación:** todas las comprobaciones se cumplen: el conteo por generación coincide con la referencia, la clave `id` es única, los nulos de `tipo_2` se conservaron, el formato largo contiene la misma información que el ancho (la suma de las seis estadísticas de cada Pokémon es su `total`) y los promedios del formato largo reproducen las tablas de la Lección 3.

## 3.4 Exportación a `data/processed/`

In [25]:
informe_limpieza.to_csv(SALIDA / 'cleaning_report.csv', index=False)
comparacion_calidad.rename_axis('metrica').to_csv(SALIDA / 'quality_comparison.csv')
pokemon_final.to_csv(SALIDA / 'pokemon_clean.csv', index=False)
estadisticas_largo.to_csv(SALIDA / 'pokemon_stats_long.csv', index=False)
promedios_generacion.to_csv(SALIDA / 'generation_averages.csv', index=False)
indicadores_cambio.to_csv(SALIDA / 'change_indicators.csv', index=False)

for archivo in sorted(SALIDA.glob('*.csv')):
    print(f'{archivo.name:28s} {archivo.stat().st_size / 1e3:8.1f} KB')

change_indicators.csv             0.6 KB
cleaning_report.csv               0.2 KB
generation_averages.csv           3.5 KB
pokemon_clean.csv                79.7 KB
pokemon_stats_long.csv          209.9 KB
quality_comparison.csv            0.2 KB


**Interpretación:** se generaron seis archivos en `data/processed/`. El más grande, `pokemon_stats_long.csv`, pesa unos 210 KB, así que todos se pueden versionar en el repositorio.

Comprobación de los archivos exportados: se leen de nuevo y se verifica que conservan las filas, las claves y los valores.

In [26]:
leido = pd.read_csv(SALIDA / 'pokemon_clean.csv')
leido_largo = pd.read_csv(SALIDA / 'pokemon_stats_long.csv')

assert leido.shape == pokemon_final.shape
assert leido['id'].is_unique
assert leido['tipo_2'].isna().sum() == 499
assert (leido[ESTADISTICAS].sum(axis=1) == leido['total']).all()
assert leido['especial'].dtype == bool
assert len(leido_largo) == 1025 * 6
assert leido_largo['valor'].sum() == leido['total'].sum()
print('Archivos exportados verificados:', leido.shape, leido_largo.shape)

Archivos exportados verificados: (1025, 17) (6150, 6)


**Interpretación:** al leer de nuevo los archivos exportados se conservan las filas, la clave, los 499 nulos de `tipo_2`, las sumas de las estadísticas y los tipos booleanos.

## 3.5 Calidad antes y después del procesamiento

In [27]:
pd.DataFrame({'antes': metricas_calidad(pokemon_crudo),
              'despues': metricas_calidad(pokemon_final)}).astype(object)

,antes,despues
filas,1025,1025
columnas,14,17
nulos_totales,499,499
filas_duplicadas,0,0
ids_repetidos,0,0
total_distinto_de_suma,0,0
estadisticas_fuera_de_rango,0,0
generacion_fuera_de_rango,0,0
tipos_invalidos,0,0


**Comparación antes y después:** las verificaciones de calidad dan cero casos tanto en el dataset crudo como en el final, porque el dato de origen ya era consistente. Lo que cambia es la estructura: de 14 a 17 columnas (`especial`, `doble_tipo`, `categoria_poder`), tipos de dato más ajustados y el formato largo y las tablas de resultados disponibles para las siguientes etapas. Los 499 nulos de `tipo_2` se mantienen porque son ausencias con significado.

## Resumen de las transformaciones

1. **Duplicados:** se verificaron filas, IDs y nombres (también tras normalizar); no había ninguno, y el paso queda en el pipeline como protección.
2. **Texto:** se comprobó que `nombre`, `tipo_1` y `tipo_2` estaban estandarizados y que los 18 tipos son válidos; la función de estandarización queda en el pipeline sin cambiar valores.
3. **Tipos de dato:** enteros a `int16` e `int8`, y `tipo_1` y `tipo_2` a `category`.
4. **Nulos:** los 499 de `tipo_2` se conservaron (Pokémon de un solo tipo).
5. **Columnas nuevas:** `especial` (legendario o mítico), `doble_tipo` y `categoria_poder`.
6. **Estructuración:** formato largo con `melt` y tablas de promedios e indicadores de cambio por generación.
7. **Validación:** comprobaciones con `assert` en el pipeline, en el dataset consolidado, en el formato largo y en los archivos exportados.

## Respuesta a la pregunta analítica

Las seis estadísticas base promedio subieron entre la generación 1 y la 9. Medido como la diferencia entre ambas generaciones, **HP es la que más cambió** (de 64,2 a 77,4: +13,2 puntos, +20,5 %), resultado que se mantiene sin legendarios ni míticos (+12,7) y con la mediana (+13). Medido como tendencia por generación, **ataque** es la que más sube (+1,63 puntos por generación). El aumento del total promedio se debe sobre todo a los Pokémon base y no a los legendarios y míticos.

## Limitaciones

- Las estadísticas son las vigentes en PokéAPI, no necesariamente las originales de cada generación.
- Hay una sola fuente y una sola forma por Pokémon (sin megaevoluciones ni formas regionales).
- Los valores atípicos se conservaron y falta contrastarlos con una segunda fuente (por ejemplo, Bulbapedia).
- Los resultados son descriptivos: las generaciones tienen tamaños distintos (de 72 a 156 Pokémon) y no se aplicó ninguna prueba estadística.